In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\keebs\OneDrive\Documentos\INTELIGENCIA ARTIFICIAL\MATERIAS 5S\PRACTICA PROFESIONALIZANTE 2\pf\bekantor-demand-intelligence


In [2]:
from src.data.load_data import load_raw_data
from src.data.clean_data import minimal_cleaning
from src.data.panel_diagnostics import (
    add_store_operation_flags,
    add_zero_taxonomy,
)

datasets = load_raw_data()
datasets = minimal_cleaning(datasets)

sales = datasets["sales"]

sales, apertura = add_store_operation_flags(sales)
sales = add_zero_taxonomy(sales)

sales_modelable = sales[
    sales["tienda_operativa"].eq(1)
    & sales["familia_surtida"].eq(1)
].copy()

print(f"Panel modelable: {len(sales_modelable):,}")
print(
    sales_modelable["date"].min().date(),
    "→",
    sales_modelable["date"].max().date(),
)

Panel modelable: 2,697,207
2013-01-01 → 2017-08-15


In [3]:
from src.models.validation import temporal_train_valid_split

train, validation, validation_start, validation_end = (
    temporal_train_valid_split(
        sales_modelable,
        validation_days=15,
    )
)

print("TRAIN")
print(train["date"].min().date(), "→", train["date"].max().date())
print(f"Registros: {len(train):,}")

print("\nVALIDACIÓN")
print(validation_start.date(), "→", validation_end.date())
print(f"Registros: {len(validation):,}")
print(f"Días: {validation['date'].nunique()}")

TRAIN
2013-01-01 → 2017-07-31
Registros: 2,671,272

VALIDACIÓN
2017-08-01 → 2017-08-15
Registros: 25,935
Días: 15


In [4]:
assert train["date"].max() < validation["date"].min()
assert validation["date"].nunique() == 15
assert len(train) + len(validation) == len(sales_modelable)

print("✓ Split temporal correcto")
print("✓ Sin solapamiento")
print("✓ 15 días de validación")

✓ Split temporal correcto
✓ Sin solapamiento
✓ 15 días de validación


In [5]:
from src.features.model_features import add_known_features

features_base = add_known_features(
    sales_modelable=sales_modelable,
    sales=sales,
    stores=datasets["stores"],
    holidays=datasets["holidays"],
)

print(f"Filas originales: {len(sales_modelable):,}")
print(f"Filas con features: {len(features_base):,}")

features_base.head()

Filas originales: 2,697,207
Filas con features: 2,697,207


,id,date,store_nbr,family,sales,onpromotion,tienda_operativa,familia_surtida,tipo_cero,year,...,type,cluster,is_holiday,is_event,is_transfer,is_additional,is_bridge,is_work_day,is_new_year_closure,event_count
0,581,2013-01-01,25,LAWN AND GARDEN,2.0,0,True,True,Venta positiva,2013,...,D,1,1,0,0,0,0,0,1,1
1,579,2013-01-01,25,HOME CARE,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
2,578,2013-01-01,25,HOME APPLIANCES,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
3,577,2013-01-01,25,HOME AND KITCHEN II,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
4,576,2013-01-01,25,HOME AND KITCHEN I,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1


In [6]:
assert len(features_base) == len(sales_modelable)

assert not features_base[
    ["city", "state", "type", "cluster"]
].isna().any().any()

assert not features_base[
    [
        "is_holiday",
        "is_event",
        "is_transfer",
        "is_additional",
        "is_bridge",
        "is_work_day",
        "is_new_year_closure",
    ]
].isna().any().any()

print("✓ Cantidad de filas preservada")
print("✓ Metadata de tiendas completa")
print("✓ Calendario completo")
print("✓ Sin utilizar ventas futuras")

✓ Cantidad de filas preservada
✓ Metadata de tiendas completa
✓ Calendario completo
✓ Sin utilizar ventas futuras


In [7]:
features_base.columns.tolist()

['id',
 'date',
 'store_nbr',
 'family',
 'sales',
 'onpromotion',
 'tienda_operativa',
 'familia_surtida',
 'tipo_cero',
 'year',
 'month',
 'day',
 'day_of_week',
 'week_of_year',
 'is_weekend',
 'city',
 'state',
 'type',
 'cluster',
 'is_holiday',
 'is_event',
 'is_transfer',
 'is_additional',
 'is_bridge',
 'is_work_day',
 'is_new_year_closure',
 'event_count']

In [8]:
from src.features.model_features import add_lag_features

features_lags = add_lag_features(
    features_base,
    lags=(1, 7, 14, 28),
)

features_lags[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
    ]
].head(40)

,date,store_nbr,family,sales,lag_1,lag_7,lag_14,lag_28
0,2013-01-01,25,LAWN AND GARDEN,2.000000,NaN,NaN,NaN,NaN
1,2013-01-01,25,HOME CARE,0.000000,NaN,NaN,NaN,NaN
2,2013-01-01,25,HOME APPLIANCES,0.000000,NaN,NaN,NaN,NaN
3,2013-01-01,25,HOME AND KITCHEN II,0.000000,NaN,NaN,NaN,NaN
4,2013-01-01,25,HOME AND KITCHEN I,0.000000,NaN,NaN,NaN,NaN
5,2013-01-01,25,HARDWARE,0.000000,NaN,NaN,NaN,NaN
6,2013-01-01,25,GROCERY II,15.000000,NaN,NaN,NaN,NaN
7,2013-01-01,25,GROCERY I,700.000000,NaN,NaN,NaN,NaN
8,2013-01-01,25,FROZEN FOODS,29.654999,NaN,NaN,NaN,NaN
9,2013-01-01,25,EGGS,46.000000,NaN,NaN,NaN,NaN


In [9]:
assert len(features_lags) == len(features_base)

for lag in [1, 7, 14, 28]:
    assert f"lag_{lag}" in features_lags.columns

print("✓ Filas preservadas")
print("✓ lag_1 creado")
print("✓ lag_7 creado")
print("✓ lag_14 creado")
print("✓ lag_28 creado")

features_lags[
    ["lag_1", "lag_7", "lag_14", "lag_28"]
].isna().mean()

✓ Filas preservadas
✓ lag_1 creado
✓ lag_7 creado
✓ lag_14 creado
✓ lag_28 creado


lag_1     0.003028
lag_7     0.006875
lag_14    0.011362
lag_28    0.020336
dtype: float64

In [10]:
from src.features.model_features import add_rolling_features

features_full = add_rolling_features(
    features_lags,
    windows=(7, 14, 28),
)

features_full[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "lag_7",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
    ]
].head(40)

,date,store_nbr,family,sales,lag_7,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,2013-01-01,25,LAWN AND GARDEN,2.000000,NaN,NaN,NaN,NaN
1,2013-01-01,25,HOME CARE,0.000000,NaN,NaN,NaN,NaN
2,2013-01-01,25,HOME APPLIANCES,0.000000,NaN,NaN,NaN,NaN
3,2013-01-01,25,HOME AND KITCHEN II,0.000000,NaN,NaN,NaN,NaN
4,2013-01-01,25,HOME AND KITCHEN I,0.000000,NaN,NaN,NaN,NaN
5,2013-01-01,25,HARDWARE,0.000000,NaN,NaN,NaN,NaN
6,2013-01-01,25,GROCERY II,15.000000,NaN,NaN,NaN,NaN
7,2013-01-01,25,GROCERY I,700.000000,NaN,NaN,NaN,NaN
8,2013-01-01,25,FROZEN FOODS,29.654999,NaN,NaN,NaN,NaN
9,2013-01-01,25,EGGS,46.000000,NaN,NaN,NaN,NaN


In [11]:
assert len(features_full) == len(features_lags)

for window in [7, 14, 28]:
    assert (
        f"rolling_mean_{window}"
        in features_full.columns
    )

print("✓ Filas preservadas")
print("✓ rolling_mean_7 creada")
print("✓ rolling_mean_14 creada")
print("✓ rolling_mean_28 creada")
print("✓ Día actual excluido")

features_full[
    [
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
    ]
].isna().mean()

✓ Filas preservadas
✓ rolling_mean_7 creada
✓ rolling_mean_14 creada
✓ rolling_mean_28 creada
✓ Día actual excluido


rolling_mean_7     0.000641
rolling_mean_14    0.000641
rolling_mean_28    0.000641
dtype: float64

In [12]:
sample = (
    features_full[
        features_full["rolling_mean_7"].notna()
    ]
    .iloc[10000]
)

mask = (
    features_base["store_nbr"].eq(
        sample["store_nbr"]
    )
    & features_base["family"].eq(
        sample["family"]
    )
    & features_base["date"].ge(
        sample["date"]
        - pd.Timedelta(days=7)
    )
    & features_base["date"].lt(
        sample["date"]
    )
)

manual_mean = features_base.loc[
    mask,
    "sales",
].mean()

print("Rolling calculada:", sample["rolling_mean_7"])
print("Rolling manual:", manual_mean)

assert np.isclose(
    sample["rolling_mean_7"],
    manual_mean,
)

print("✓ Rolling verificada sin usar el día actual")

Rolling calculada: 0.0
Rolling manual: 0.0
✓ Rolling verificada sin usar el día actual
